# Train HeatGeo from GitHub (Colab)

Notebook này luôn đồng bộ code mới nhất từ branch `nqd_mass_geom_loss` trước khi cài dependencies và train. Chọn một teacher–student pair, chọn `ROW_MODE`, rồi dùng **Runtime → Run all**.

## `ROW_MODE`: cách `L_row` chọn row

| Mode | Row được chọn thế nào | Trọng số `nu_B` | Walk HP |
|---|---|---|---:|
| `walk` | non-backtracking teacher walk từ mỗi anchor | visit count | 2 |
| `closure_u` | mọi pool column được teacher chọn (diffusion support) | uniform | 0 |
| `closure_m` | như trên | exposed teacher mass `m_B(j)` | 0 |

Walk chỉ dùng để *chọn* row — trajectory không bao giờ là target — nên `NUM_WALKS` và `WALK_LENGTH` không xuất hiện trong objective. Hai closure mode promote thẳng các column mà `L_rel` đã trả tiền để encode, nên bỏ được cả hai. Batch anchor bị loại khỏi row set vì `L_rel` đã match transition row của chúng ở scale r=1.

## Kết quả (`qwen3_0_6b_to_minilmv2_h384`, `ROW_WEIGHT=1.0`, `ROW_START_EPOCH=2`, seed 42)

| Mode | Avg | Walk HP | `row_count`/batch |
|---|---:|---:|---:|
| `walk` | 74.88 | 2 | 897 |
| **`closure_u`** | **74.86** | **0** | 843 |
| `closure_m` | 74.76 | 0 | 843 |

`closure_u` giữ nguyên hiệu năng của walk (−0.02, dưới mức nhiễu single-seed) và bỏ được cả hai hyperparameter — đây là default hiện tại. Hai cơ chế cấp gần như cùng lượng supervision (843 vs 897 row), nên điểm số trùng nhau là hợp lý.

`closure_m` thua 0.10 vì mass weighting dồn trọng số về các row nằm sâu trong neighborhood của anchor, tức phần supervision gần trùng với target r=1 mà `L_rel` đã cấp. Kiểm chứng bằng `row_eff_count` so với `row_count` trong log mỗi epoch: dưới `closure_u` hai giá trị bằng nhau, dưới `closure_m` thì `row_eff_count` thấp hơn.

## Bỏ nốt `ROW_START_EPOCH`

| `ROW_MODE` | `ROW_START_EPOCH` | Avg |
|---|---:|---:|
| `walk` | 2 | 74.88 |
| `closure_u` | 2 | 74.86 |
| **`closure_u`** | **1** | **74.82** |

Chênh 0.04, trong nhiễu single-seed. Diagnostics epoch 1 của run start-at-1 giống hệt các epoch sau (`row_count=844`, `row_exposed_mass=0.4407`), và `loss_rel` cuối còn thấp hơn run start-at-2. `ROW_START_EPOCH=1` là default mới, tức knob này trở nên inert — **`L_row` chỉ còn đúng một hyperparameter là `ROW_WEIGHT`**.

**Cảnh báo:** 74.88 → 74.86 → 74.82 đều trong nhiễu nếu xét từng bước, nhưng ba bước cùng chiều đi xuống. Single seed không phân biệt được "nhiễu" với "chi phí nhỏ tích lũy". Trước khi chốt số cho paper phải chạy **3 seed** cho `closure_u` + start-1 và so với 3 seed của walk.

## Run plan hướng 75.2+ (mỗi dòng một run, baseline 74.82 = closure_u/w1.0/e1)

| Run | Thay đổi so với baseline | Giả thuyết |
|---|---|---|
| R1 | `ROW_WEIGHT = 2.0` | bảng tuning monotone 0.5→74.63, 1.0→74.88 và chưa từng thử >1.0 |
| R2 | `ROW_AMBIENT = True` | 843 row/batch hiện không có ambient calibration; exposed mass chỉ 0.44 |
| R3 | `EPOCHS = 8` | mọi KL vẫn đang giảm ở epoch 5; `loss_excess` = 0.66; student_top1 0.20 vs teacher 0.33 |
| R4 | `ROW_MODE = "closure_ht"` | nấc bias⁰ của thang trọng số; kỳ vọng nhỏ, chạy sau cùng |

Chạy R1–R3 trước (độc lập), rồi stack các arm thắng và chạy 3 seed cho combo. Theo dõi `row_amb_kl` (R2) và `row_eff_count` (R4) trong log.

Khi `ROW_MODE` khác `walk`, `NUM_WALKS` bị ép về 0 phía config, và truyền `--num_walks` cùng một closure mode sẽ báo lỗi thay vì bị bỏ qua âm thầm. Lưu ý so sánh không hoàn toàn isolated: bỏ walk cũng khôi phục các uniform negative mà walk-visited node từng thay thế trong candidate draw, nên `L_rel` đổi nhẹ.

> Lưu ý: cell đồng bộ code dùng `git reset --hard` bên trong `/content/embedding-kd`, nên các chỉnh sửa tracked cục bộ trong bản clone Colab sẽ bị bỏ. Cache và output untracked không bị xóa. Pair Qwen3-4B → BERT-base cần GPU có VRAM lớn.


In [13]:
#@title 1. Cấu hình experiment
REPO_URL = "https://github.com/Savoxism/embedding-kd.git"
BRANCH = "nqd_mass_geom_loss"

PAIR_KEY = "qwen3_0_6b_to_minilmv2_h384" #@param
# ["qwen3_0_6b_to_minilmv2_h384", "bge_m3_to_minilmv2_h768", "qwen3_4b_to_bert_base"]

# Cách L_row chọn row. closure_u là arm đã chọn: ngang walk (74.86 vs 74.88)
# nhưng bỏ hẳn NUM_WALKS/WALK_LENGTH. Xem bảng ở cell markdown đầu notebook.
ROW_MODE = "closure_u" #@param ["closure_u", "closure_ht", "closure_m", "walk"]
# closure_ht: cùng row set với closure_u nhưng nu(j) ∝ 1/c_B(j) (số anchor đã
# chọn column j) — nấc "bias⁰" của thang bias: m=bias², u=bias¹, ht≈bias⁰.
ROW_WEIGHT = 1.0 #@param {type:"number"}
# Ambient r=0 cho mỗi promoted row (dense teacher-similarity trên pool tại
# direct_temp, trọng số buộc omega_amb = omega_1 như anchor — không thêm knob).
# Động lực: row_exposed_mass = 0.44, restricted target chỉ thấy 44% mass thật.
ROW_AMBIENT = False #@param {type:"boolean"}
ROW_START_EPOCH = 1 #@param {type:"integer"}

# Chỉ có tác dụng khi ROW_MODE == "walk"; closure mode bỏ qua hai dòng này.
NUM_WALKS = 4 #@param {type:"integer"}
WALK_LENGTH = 4 #@param {type:"integer"}

RUN_NAME = "row_loss_run" #@param {type:"string"}

BATCH_SIZE = 64 #@param {type:"integer"}
EPOCHS = 5 #@param {type:"integer"}
LEARNING_RATE = 2e-5 #@param {type:"number"}
MAX_LENGTH = 256 #@param {type:"integer"}
SEED = 42 #@param {type:"integer"}
NUM_WORKERS = 4 #@param {type:"integer"}
TRAIN_DATA = "data/train_set/merged_3_data_5k_each.csv" #@param {type:"string"}

USE_WANDB = False #@param {type:"boolean"}
WANDB_PROJECT = "iclr-mdd-heatgeo" #@param {type:"string"}
WANDB_MODE = "offline" #@param ["online", "offline", "disabled"]
FINAL_WEIGHTS_ONLY = True #@param {type:"boolean"}
REQUIRE_GPU = True #@param {type:"boolean"}

USE_GOOGLE_DRIVE = False #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/embedding-kd-runs" #@param {type:"string"}

# Thêm CLI flags nếu cần, ví dụ: --graph_k 100 --diffusion_quota 20
EXTRA_ARGS = "" #@param {type:"string"}

ROW_MODES = ("walk", "closure_u", "closure_m", "closure_ht")
assert ROW_MODE in ROW_MODES, f"ROW_MODE phải thuộc {ROW_MODES}"
assert ROW_WEIGHT >= 0, "ROW_WEIGHT phải không âm"
assert NUM_WALKS >= 0, "NUM_WALKS phải không âm"
assert WALK_LENGTH > 0, "WALK_LENGTH phải dương"
assert ROW_START_EPOCH >= 1, "ROW_START_EPOCH phải bắt đầu từ 1"
assert BATCH_SIZE > 0 and EPOCHS > 0 and MAX_LENGTH > 0

# Mỗi variant ghi vào thư mục riêng để các arm của ablation không đè lên nhau.
# Phải mang đủ mọi knob đang quét: chỉ có ROW_MODE và SEED là chưa đủ — hai run
# closure_u khác nhau ở ROW_START_EPOCH sẽ ghi đè lẫn nhau.
_amb = "_amb" if ROW_AMBIENT else ""
RUN_TAG = f"{RUN_NAME}_{ROW_MODE}{_amb}_w{ROW_WEIGHT:g}_e{ROW_START_EPOCH}_seed{SEED}"


In [14]:
# 2. Clone lần đầu; các lần Run all sau luôn fetch/reset/pull branch mới nhất
from pathlib import Path
import subprocess

REPO_DIR = Path("/content/embedding-kd")

def git(*args):
    command = ["git", "-C", str(REPO_DIR), *args]
    print("+", " ".join(command))
    subprocess.run(command, check=True)

if (REPO_DIR / ".git").is_dir():
    git("remote", "set-url", "origin", REPO_URL)
    # Bỏ thay đổi tracked trong clone Colab để luôn checkout được remote head.
    git("reset", "--hard")
    git("fetch", "--prune", "origin")
    git("checkout", "-B", BRANCH, f"origin/{BRANCH}")
    git("reset", "--hard", f"origin/{BRANCH}")
    git("pull", "--ff-only", "origin", BRANCH)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"{REPO_DIR} tồn tại nhưng không phải Git repository")
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print(f"Ready: {BRANCH}@{commit[:12]}")


+ git -C /content/embedding-kd remote set-url origin https://github.com/Savoxism/embedding-kd.git
+ git -C /content/embedding-kd reset --hard
+ git -C /content/embedding-kd fetch --prune origin
+ git -C /content/embedding-kd checkout -B nqd_mass_geom_loss origin/nqd_mass_geom_loss
+ git -C /content/embedding-kd reset --hard origin/nqd_mass_geom_loss
+ git -C /content/embedding-kd pull --ff-only origin nqd_mass_geom_loss
Ready: nqd_mass_geom_loss@07e7a879ab48


In [15]:
# 3. Cài/đồng bộ dependencies theo code vừa pull
import os
import sys

os.chdir(REPO_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
print("Dependencies are ready.")


Dependencies are ready.


In [16]:
# 4. Resolve model pair, GPU và nơi lưu artifacts
import torch

PAIR_CONFIGS = {
    "qwen3_0_6b_to_minilmv2_h384": {
        "teacher": "Qwen/Qwen3-Embedding-0.6B",
        "student": "nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base",
        "pooling": "last_token",
    },
    "bge_m3_to_minilmv2_h768": {
        "teacher": "BAAI/bge-m3",
        "student": "nreimers/MiniLMv2-L6-H768-distilled-from-BERT-Base",
        "pooling": "cls",
    },
    "qwen3_4b_to_bert_base": {
        "teacher": "Qwen/Qwen3-Embedding-4B",
        "student": "google-bert/bert-base-uncased",
        "pooling": "last_token",
    },
}
pair = PAIR_CONFIGS[PAIR_KEY]

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Không tìm thấy CUDA GPU. Trong Colab chọn Runtime → Change runtime type → GPU.")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 2**30
    print(f"GPU: {props.name} ({vram_gb:.1f} GiB)")
    if PAIR_KEY == "qwen3_4b_to_bert_base" and vram_gb < 35:
        print("WARNING: Qwen3-4B ở cấu hình hiện tại có thể OOM trên GPU dưới khoảng 35 GiB.")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    artifact_root = Path(DRIVE_ROOT)
else:
    artifact_root = REPO_DIR

# cache_dir không mang RUN_TAG: teacher embeddings và HeatGeo graph không phụ
# thuộc row mode, nên ba variant dùng chung một cache và chỉ build graph một lần.
cache_dir = artifact_root / "cache" / "heatgeo" / PAIR_KEY
log_dir = artifact_root / "logs" / "heatgeo" / PAIR_KEY
save_dir = artifact_root / "models" / "heatgeo" / PAIR_KEY / RUN_TAG
weights_dir = artifact_root / "models" / "heatgeo_weights" / PAIR_KEY / RUN_TAG
for path in (cache_dir, log_dir, save_dir, weights_dir):
    path.mkdir(parents=True, exist_ok=True)

ROW_MODE_NOTES = {
    "walk": f"non-backtracking walk ({NUM_WALKS} walks x {WALK_LENGTH} steps)",
    "closure_u": "teacher-selected columns promoted to rows, uniform weight",
    "closure_m": "teacher-selected columns promoted to rows, exposed-mass weight",
}

print(f"Teacher: {pair['teacher']}")
print(f"Student: {pair['student']}")
print(f"Pooling: {pair['pooling']}")
print(f"Objective: L_rel + {ROW_WEIGHT} * L_row")
print(f"Row mode: {ROW_MODE} — {ROW_MODE_NOTES[ROW_MODE]}")
print(f"Row starts at epoch: {ROW_START_EPOCH}")
print(f"Outputs: {save_dir}")


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB)
Teacher: Qwen/Qwen3-Embedding-0.6B
Student: nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base
Pooling: last_token
Objective: L_rel + 1.0 * L_row
Row mode: closure_u — teacher-selected columns promoted to rows, uniform weight
Row starts at epoch: 1
Outputs: /content/embedding-kd/models/heatgeo/qwen3_0_6b_to_minilmv2_h384/row_loss_run_closure_u_seed42


In [17]:
# 5. Train HeatGeo
import shlex

command = [
    sys.executable, "main.py",
    "--method", "heatgeo",
    "--train_data", TRAIN_DATA,
    "--student_model", pair["student"],
    "--teacher_model", pair["teacher"],
    "--pooling_method", pair["pooling"],
    "--row_mode", ROW_MODE,
    "--row_ambient", str(int(ROW_AMBIENT)),
    "--row_weight", str(ROW_WEIGHT),
    "--row_start_epoch", str(ROW_START_EPOCH),
    "--batch_size", str(BATCH_SIZE),
    "--epochs", str(EPOCHS),
    "--lr", str(LEARNING_RATE),
    "--max_length", str(MAX_LENGTH),
    "--seed", str(SEED),
    "--num_workers", str(NUM_WORKERS),
    "--cache_path", str(cache_dir / "teacher_train.pt"),
    "--heatgeo_cache_path", str(cache_dir / "graph.pt"),
    "--heatgeo_log_dir", str(log_dir),
    "--save_dir", str(save_dir),
    "--weights_dir", str(weights_dir),
    "--wandb_project", WANDB_PROJECT,
    "--wandb_run_name", f"{PAIR_KEY}_{RUN_TAG}",
    "--wandb_mode", WANDB_MODE,
]
# Chỉ walk mode mới nhận hai flag này. Truyền chúng ở closure mode sẽ bị config ép
# về 0 và chỉ làm log gây hiểu nhầm là walk vẫn đang chạy.
if ROW_MODE == "walk":
    command += ["--num_walks", str(NUM_WALKS), "--walk_length", str(WALK_LENGTH)]
if FINAL_WEIGHTS_ONLY:
    command.append("--final_weights_only")
if not USE_WANDB:
    command.append("--no_wandb")
if EXTRA_ARGS.strip():
    command.extend(shlex.split(EXTRA_ARGS))

env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("+", shlex.join(command))
process = subprocess.Popen(
    command,
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Training failed with exit code {return_code}")


+ /usr/bin/python3 main.py --method heatgeo --train_data data/train_set/merged_3_data_5k_each.csv --student_model nreimers/MiniLMv2-L6-H384-distilled-from-BERT-Base --teacher_model Qwen/Qwen3-Embedding-0.6B --pooling_method last_token --row_mode closure_u --row_weight 1.0 --row_start_epoch 1 --batch_size 64 --epochs 5 --lr 2e-05 --max_length 256 --seed 42 --num_workers 4 --cache_path /content/embedding-kd/cache/heatgeo/qwen3_0_6b_to_minilmv2_h384/teacher_train.pt --heatgeo_cache_path /content/embedding-kd/cache/heatgeo/qwen3_0_6b_to_minilmv2_h384/graph.pt --heatgeo_log_dir /content/embedding-kd/logs/heatgeo/qwen3_0_6b_to_minilmv2_h384 --save_dir /content/embedding-kd/models/heatgeo/qwen3_0_6b_to_minilmv2_h384/row_loss_run_closure_u_seed42 --weights_dir /content/embedding-kd/models/heatgeo_weights/qwen3_0_6b_to_minilmv2_h384/row_loss_run_closure_u_seed42 --wandb_project iclr-mdd-heatgeo --wandb_run_name qwen3_0_6b_to_minilmv2_h384_row_loss_run_closure_u_seed42 --wandb_mode offline --fin

In [18]:
# 6. Xem artifacts và các metrics cuối
import json

print("Checkpoints:", save_dir)
print("Weights:", weights_dir)
metrics_path = save_dir / "metrics.jsonl"
if metrics_path.exists():
    records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    print(json.dumps(records[-1], indent=2, ensure_ascii=False) if records else "metrics.jsonl is empty")
else:
    print("Không tìm thấy metrics.jsonl")


Checkpoints: /content/embedding-kd/models/heatgeo/qwen3_0_6b_to_minilmv2_h384/row_loss_run_closure_u_seed42
Weights: /content/embedding-kd/models/heatgeo_weights/qwen3_0_6b_to_minilmv2_h384/row_loss_run_closure_u_seed42
{
  "method": "heatgeo",
  "seed": 42,
  "test": {
    "avg": 74.82,
    "avg_in": 67.94,
    "avg_out": 78.26,
    "classification": {
      "data/test_set/banking77_test.csv": {
        "accuracy": 0.8813394018205462,
        "f1": 0.8817056038616747
      },
      "data/test_set/emotion_test.csv": {
        "accuracy": 0.7089627391742196,
        "f1": 0.6186762633601979
      },
      "data/test_set/tweet_test.csv": {
        "accuracy": 0.7016317016317016,
        "f1": 0.7056083418100836
      }
    },
    "pair": {
      "data/test_set/mrpc_test.csv": {
        "accuracy": 0.7084057971014492,
        "average_precision": 0.8220035159636716,
        "best_threshold": 0.9246231155778895,
        "f1": 0.6290117969694987,
        "precision": 0.6707307423202505,
   